In [2]:
import pandas as pd
import numpy as np
import os
import warnings

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)


# -----------------------------
# Experiment name
# -----------------------------
experiment_name = "exp11_elasticnet_kfold_ensemble_fixed_20260323"


# -----------------------------
# Load data
# -----------------------------
def load_data():

    print("Loading data...")

    train = pd.read_csv("../data/train.csv", encoding="cp932")
    test = pd.read_csv("../data/test.csv", encoding="cp932")

    print("Train shape:", train.shape)
    print("Test shape:", test.shape)

    return train, test


# -----------------------------
# Prepare features
# -----------------------------
def prepare_features(train, test):

    spectral_cols = [
        c for c in train.columns
        if c not in ["sample number", "species number", "樹種", "含水率"]
    ]

    X = train[spectral_cols].values
    y = train["含水率"].values
    X_test = test[spectral_cols].values

    print("Number of spectral features:", X.shape[1])

    return X, y, X_test


# -----------------------------
# Scale features
# -----------------------------
def scale_features(X, X_test):

    scaler = StandardScaler()

    X_scaled = scaler.fit_transform(X)
    X_test_scaled = scaler.transform(X_test)

    print("Feature scaling complete")

    return X_scaled, X_test_scaled


# -----------------------------
# Cross validation
# -----------------------------
def cross_validate(X, y):

    print("\nRunning 5-fold cross-validation...")

    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    rmse_scores = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):

        print(f"Training fold {fold+1}")

        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        model = ElasticNet(
            alpha=0.1,
            l1_ratio=0.5,
            max_iter=10000,
            random_state=42
        )

        model.fit(X_train, y_train)

        preds = model.predict(X_val)

        rmse = np.sqrt(mean_squared_error(y_val, preds))
        rmse_scores.append(rmse)

        print("Fold RMSE:", rmse)

    print("\nMean CV RMSE:", np.mean(rmse_scores))


# -----------------------------
# Ensemble prediction
# -----------------------------
def kfold_ensemble(X, y, X_test):

    print("\nTraining KFold ElasticNet ensemble...")

    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    test_preds = np.zeros(X_test.shape[0])

    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):

        print(f"Training ensemble model {fold+1}")

        X_train = X[train_idx]
        y_train = y[train_idx]

        model = ElasticNet(
            alpha=0.1,
            l1_ratio=0.5,
            max_iter=10000,
            random_state=42
        )

        model.fit(X_train, y_train)

        preds = model.predict(X_test)

        test_preds += preds / 5

    print("Sample ensemble predictions:", test_preds[:10])

    return test_preds


# -----------------------------
# Save submission
# -----------------------------
def save_submission(test, preds):

    os.makedirs("../submissions", exist_ok=True)

    submission = pd.DataFrame({
        "sample number": test["sample number"],
        "含水率": preds
    })

    output_path = f"../submissions/{experiment_name}.csv"

    submission.to_csv(output_path, index=False, header=False)

    print("\nSubmission saved to:", output_path)

    check = pd.read_csv(output_path, header=None)
    print(check.head())


# -----------------------------
# Main
# -----------------------------
def main():

    train, test = load_data()

    X, y, X_test = prepare_features(train, test)

    X_scaled, X_test_scaled = scale_features(X, X_test)

    cross_validate(X_scaled, y)

    preds = kfold_ensemble(X_scaled, y, X_test_scaled)

    save_submission(test, preds)


main()

Loading data...
Train shape: (1322, 1559)
Test shape: (550, 1558)
Number of spectral features: 1555
Feature scaling complete

Running 5-fold cross-validation...
Training fold 1
Fold RMSE: 18.118939325203428
Training fold 2
Fold RMSE: 20.272005183550387
Training fold 3
Fold RMSE: 21.78801303595941
Training fold 4
Fold RMSE: 18.4479342796035
Training fold 5
Fold RMSE: 21.24909937729737

Mean CV RMSE: 19.97519824032282

Training KFold ElasticNet ensemble...
Training ensemble model 1
Training ensemble model 2
Training ensemble model 3
Training ensemble model 4
Training ensemble model 5
Sample ensemble predictions: [171.78947113 177.28008722 180.00562138 177.9021691  173.24606933
 163.63765178 154.24734892 146.41953976 139.23142239 133.28474211]

Submission saved to: ../submissions/exp11_elasticnet_kfold_ensemble_fixed_20260323.csv
    0           1
0  95  171.789471
1  96  177.280087
2  97  180.005621
3  98  177.902169
4  99  173.246069
